# Chapter 2 examples: finding the best policy in one pass

This notebook runs the examples of [Chapter 2](https://github.com/thduynguyen/gtsam/blob/feature/semiringfactor/gtsam/semiring/doc/chapter02.md): the best
policy of the two examples of Chapter 1, found in one backward pass by taking
a maximum over each action where Chapter 1 took an average.

GTSAM Copyright 2010, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/thduynguyen/gtsam/blob/feature/semiringfactor/gtsam/semiring/doc/chapter02_examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    # The semiring module is not in a released GTSAM yet, so install a wheel
    # built from the feature branch.
    %pip install --quiet numpy pyparsing plotly
    %pip install --quiet --no-index --no-deps --find-links https://github.com/thduynguyen/gtsam/releases/expanded_assets/semiring-wheels gtsam-develop
except ImportError:
    pass  # Not running on Colab, do nothing

In [2]:
import numpy as np

from gtsam import (
    DecisionTreeFactor,
    DiscreteValues,
    HessianFactor,
    JacobianFactor,
    Ordering,
    SemiringDiscreteFactor,
    SemiringGaussianFactor,
    noiseModel,
)
from gtsam.symbol_shorthand import A, S, U, X

np.set_printoptions(precision=4, suppress=True)


def ordering(*keys):
    """An elimination ordering holding the given keys."""
    result = Ordering()
    for key in keys:
        result.push_back(key)
    return result


def table(factor, keys):
    """Read a discrete table into an array indexed in the order of keys."""
    result = np.zeros([cardinality for _, cardinality in keys])
    for index in np.ndindex(*result.shape):
        values = DiscreteValues()
        for (key, _), value in zip(keys, index):
            values[key] = value
        result[index] = factor(values)
    return np.round(result, 9) + 0.0  # rounds off noise, avoids printing -0


# The track example of Chapter 1: tables, and how they are lifted.
def state(t):
    """Discrete key of the cell after t moves: (key, number of values)."""
    return (S(t), 3)


def action(t):
    """Discrete key of move t, where 0 is Left and 1 is Right."""
    return (A(t), 2)


def probability(keys, values):
    """Lift a probability table to the semiring factor (p, 0)."""
    return SemiringDiscreteFactor(
        DecisionTreeFactor(keys, np.ravel(values).tolist()))


def reward(keys, values):
    """Lift a reward table to the semiring factor (1, r)."""
    return SemiringDiscreteFactor.Reward(
        DecisionTreeFactor(keys, np.ravel(values).tolist()))


# The tables of the problem. The first key of a table varies slowest.
start = [0.5, 0.5, 0.0]                          # p(s0)
coin_flip = [[0.5, 0.5]] * 3                     # pi(a | s)
slippery = [[[1.0, 0.0, 0.0], [0.2, 0.8, 0.0]],  # p(s' | s, a)
            [[0.8, 0.2, 0.0], [0.0, 0.2, 0.8]],
            [[0.0, 0.8, 0.2], [0.0, 0.0, 1.0]]]
move_reward = [[0.0, -1.0]] * 3                  # r(s, a)
final_reward = [0.0, 0.0, 10.0]                  # r(s2)

# The line example of Chapter 1: helpers to lift densities and penalties.
I = np.eye(1)
zero = np.zeros(1)


def variance(v):
    """A scalar Gaussian noise model with the given variance."""
    return noiseModel.Isotropic.Variance(1, v)


def gaussian(*args):
    """Lift a Gaussian factor to the semiring factor (p, 0)."""
    return SemiringGaussianFactor(JacobianFactor(*args))


def penalty(key):
    """Lift the penalty z^2 on one variable to the reward (1, -z^2)."""
    return SemiringGaussianFactor.Cost(HessianFactor(key, 2 * I, zero, 0.0))

## Tabular case: the track (Section 2)

There is no policy factor. At each step the next state is eliminated by
expectation, and the action by maximum.

The module has no built-in maximum yet. The best move of each cell is picked
from the table of Q, and put back as a policy that always takes it; summing
that policy out then substitutes the best move, which is the maximum.

In [3]:
value = reward([state(2)], final_reward)  # (1, V*_2)
for t in reversed(range(2)):
    keys = [state(t), action(t)]

    # Eliminate the next state by expectation.
    dynamics = probability(keys + [state(t + 1)], slippery)
    phi = (dynamics * value).sum(ordering(S(t + 1)))

    # Multiply with the reward: (1, Q*_t). There is no policy factor.
    bucket = reward(keys, move_reward) * phi
    Q = table(bucket.value(), keys)

    # Eliminate the action by max. The best move of each cell is lifted as a
    # one-hot policy table, so that summing it out substitutes that move.
    best_move = Q.argmax(axis=1)
    best = probability(keys, np.eye(2)[best_move])
    value = (best * bucket).sum(ordering(A(t)))  # (1, V*_t)

    print(f"Q*_{t}: rows s, columns L and R")
    print(Q)
    print("best move in cells 0, 1, 2:", ["LR"[move] for move in best_move])
    print(f"V*_{t}:", table(value.value(), [state(t)]))
    print()

best_return = (probability([state(0)], start) * value).expectation()
print("J* =", round(best_return, 6))

Q*_1: rows s, columns L and R
[[ 0. -1.]
 [ 0.  7.]
 [ 2.  9.]]
best move in cells 0, 1, 2: ['L', 'R', 'R']
V*_1: [0. 7. 9.]

Q*_0: rows s, columns L and R
[[0.  4.6]
 [1.4 7.6]
 [7.4 8. ]]
best move in cells 0, 1, 2: ['R', 'R', 'R']
V*_0: [4.6 7.6 8. ]

J* = 6.1


## LQR: the line (Section 3)

First the Riccati recursion, written out directly.

In [4]:
# The classic recursion for the line example:
# F = B = C_x = C_u = C_T = 1, noise variance 0.5, x0 ~ N(2, 1).
F = B = C_x = C_u = 1.0
P, beta = 1.0, 0.0  # P_2 = C_T
for t in reversed(range(2)):
    K = B * P * F / (C_u + B * P * B)
    beta = beta + P * 0.5
    P = C_x + F * P * F - (F * P * B)**2 / (C_u + B * P * B)
    print(f"t = {t}: K = {K:.3f}, P = {P:.3f}, beta = {beta:.3f}")
print("J* =", round(-(P * (2.0**2 + 1.0) + beta), 6))

t = 1: K = 0.500, P = 1.500, beta = 0.500
t = 0: K = 0.600, P = 1.600, beta = 1.250
J* = -9.25


Then the same numbers from elimination on the factor graph. The maximum
over a move solves a linear equation for the gain, and the best policy is put
back as a hard constraint u = -K x.

In [5]:
value = penalty(X(2))  # (1, V_2)
for t in reversed(range(2)):
    # Eliminate the next state by expectation.
    dynamics = gaussian(X(t + 1), I, X(t), -I, U(t), -I, zero, variance(0.5))
    phi = dynamics.multiply(value).sum(ordering(X(t + 1)))

    # Multiply with the rewards: values add, giving (1, Q_t).
    bucket = penalty(X(t)).multiply(penalty(U(t))).multiply(phi)

    # Maximize over u: solve H_uu u = -H_ux x for the gain.
    Q = bucket.value()  # a quadratic on u and x
    keys, H = list(Q.keys()), Q.information()
    u, x = keys.index(U(t)), keys.index(X(t))
    K = H[u, x] / H[u, u]

    # The best policy u = -K x as a factor, then eliminate u.
    best = gaussian(U(t), I, X(t), K * I, zero, noiseModel.Constrained.All(1))
    value = best.multiply(bucket).sum(ordering(U(t)))  # (1, V_t)

    V = value.value()  # 0.5 G x^2 + 0.5 f = -(P x^2 + beta)
    print(f"t = {t}: K = {K:.3f}, P = {-0.5 * V.information()[0, 0]:.3f}, "
          f"beta = {-0.5 * V.constantTerm():.3f}")

# Eliminate the first state by expectation: the best expected return.
prior = gaussian(X(0), I, np.array([2.0]), variance(1.0))
print("J* =", round(prior.multiply(value).expectation(), 6))

t = 1: K = 0.500, P = 1.500, beta = 0.500
t = 0: K = 0.600, P = 1.600, beta = 1.250
J* = -9.25
